# Take data from CSV and add to items or make new item

In [ ]:
import pandas as pd
import re
from datetime import datetime
import wikibaseintegrator
from wikibaseintegrator import WikibaseIntegrator, wbi_helpers, wbi_login, datatypes
from wikibaseintegrator.wbi_config import config as wbi_config
from wikibaseintegrator.entities import ItemEntity
from wikibaseintegrator.models import Qualifiers, References, Reference
from wikibaseintegrator.wbi_enums import ActionIfExists
import logging
import json
import random
from pathlib import Path
from typing import Optional, Union
from dataclasses import dataclass, field
from __future__ import annotations
from enum import Enum
import numpy as np

collections_dir = Path("../wikidata/metadata_collections/")

with Path('../authorization.json').open(mode='r') as authorization_file:
    authorization = json.load(authorization_file)

USER_AUTH = authorization['user_auth']
USERNAME = authorization['username']
PASSWORD = authorization['password']
CONSUMER_TOKEN = authorization['consumer_token']
CONSUMER_SECRET = authorization['consumer_secret']

BOTNAME = authorization['botname']
    

log_file = str(Path('logs/Make_WikibaseItem_from_Metadata.log'))
logging.basicConfig(filename=log_file, force=True,
                    format='%(asctime)s %(message)s', 
                    datefmt='%Y/%m/%d %H:%M:%S',
                    encoding='utf-8', 
                    level=logging.DEBUG)

logger = logging.getLogger('Make-Items')
logger.debug('Start logging')


wbi_config['USER_AGENT'] = f'{BOTNAME} (https://www.wikidata.org/wiki/User:{USERNAME})'
wbi_config['MEDIAWIKI_API_URL'] = 'https://test.wikidata.org/w/api.php'

PROPS = {'instance_of':'P31',
        'author':'P50',
         'title':'P1476',
         'has_edition':'P747',
         'edition_of': 'P629',
         'based_on' : 'P144',
         'language':'P407',
         'publication_date':'P577',
         'work_available_at_URL':'P953',
         'has_edition_or_translation' : 'P747',
         'project_gb_ebook_id' : 'P2034',
         }
ENTITIES = {
   'literary_work':'Q7725634', 
   'edition' : 'Q3331189',
   'German':'Q188',
   'Hugo_Ball':'Q70989'
}

## Load data from CSV (containing data from Openrefine)

In [ ]:
fiction = pd.read_csv(Path(collections_dir, 'de_fiction_metadata_2025-11-14T18.csv'), index_col=0, dtype={'gutenberg_id':str})
fiction

## Log in to Wikidata

In [ ]:
login_instance = wbi_login.Login(user=USER_AUTH, password=PASSWORD)

wbi = WikibaseIntegrator(login=login_instance)

randhex = "{:x}".format(random.randrange(0, 2**48))
EDIT_SUMMARY=f'{BOTNAME}: test ([[:toolforge:editgroups/b/CB/{randhex}|details]])'

# Start making new Entities

In [ ]:
def make_sparql_authors_works(author_qid):
    return '''SELECT ?item ?title ?year (COUNT(?edition) as ?count)
WHERE {
    # is a literary work
  ?item wdt:P31 wd:Q7725634 .
  # author is ...
  ?item wdt:P50 wd:''' + author_qid + ''' . 
  OPTIONAL {
  ?edition wdt:P629 ?item . }
  OPTIONAL {
    ?item wdt:P1476 ?title .} 
  OPTIONAL {
    ?item wdt:P577 ?year . }
  }
GROUP BY ?item ?title ?year
ORDER BY DESC(?count)'''




In [ ]:
hennings_qid = 'Q76815'
works_by_Hennings = wbi_helpers.execute_sparql_query(make_sparql_authors_works(hennings_qid), 
                                          user_agent=wbi_config['USER_AGENT'])
works_by_Hennings

## Let's upload our data for Emmy Hennings

In [ ]:
view = fiction[fiction['author'].str.contains("Hennings")]
view

In [ ]:
author_qid = hennings_qid
author_entity = wbi.item.get(entity_id=author_qid)
author_entity.get_json()

In [ ]:
def date_tag_precise():
    return datetime.today().strftime("%Y-%m-%dT%H:%M:%S")

class Author():
    def __init__(self, item = None, qid=None, name = None, works = None, editions = None):
        self._item = item
        self._qid = qid
        self._name = name
        self._works = works
        self._editions = editions
    
    @classmethod
    def from_item(self, item: wikibaseintegrator.entities.item.ItemEntity):
        return Author(item=item)
    
    def get_name(self, language='mul'):
        if self._item:
            if self._item.labels.get(language):
                return self._item.labels.get(language).value
            if self._item.labels.get('mul'):
                return self._item.labels.get('mul').value
            if self._name:
                return self._name
            else:
                if self._quid:
                    raise Exception(f'No name available for {self._qid}')
                if self.item:
                    raise Exception(f'No name available for {self._item}')
                raise Exception(f'No name available for {self}')
        else:
            return self.name

            
        

In [ ]:
type(author_entity)

In [ ]:
author_entity.labels.get('en').value

In [ ]:
author = Author.from_item(author_entity)
author.get_name('de')

## Make a new work entry

In [ ]:
work = view.iloc[1]
work

In [ ]:
# CURRENTLY WE ONLY DO SINGLE AUTHORED WORKS
# Otherwise, parsing the names is a hassle and we have only a handful of works with more than one author,
# ...And all of them have data inconsistency issues that need to be adressed first

def format_year(year : int):
    return datetime(year, 1, 1).strftime("+%Y-%m-%dT%H:%M:%SZ")

def make_work(author_qid : str, author : Author, title : str, 
              year : int = None, url : Optional[str] = None): 
    language = ENTITIES['German']

    formatted_year = format_year(year)

    new_work = wbi.item.new()

    new_work.labels.set('de', title)
    # Set a default label too
    new_work.labels.set('mul', title)
    new_work.descriptions.set('en', f'Literary work of fiction by {author.get_name('en')}')
    new_work.descriptions.set('de', f'Fiktionales literarisches Werk von {author.get_name('de')}')
    
    new_work.claims.add([
        datatypes.Item(value=ENTITIES['literary_work'], prop_nr=PROPS['instance_of']), 
        datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
        #
        # NEED TO CHECK IF IT HAS OTHER AUTHORS
        #
        datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
        datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
        datatypes.Item(value=language, prop_nr=PROPS['language'])
                    ])
    if url:
        new_work.claims.add([
            datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                        ])
    return new_work

In [ ]:
title = work['title'].strip()
title

In [ ]:
print(author_name)
author

In [ ]:
year = work['year'].item()

# Demonstrate
formatted_year = format_year(year)
formatted_year

In [ ]:
#author_name = work['author'].strip() 

new_work = make_work(author_qid = author_qid, author=author, title = title, 
                    year = year, url=url)
new_work.get_json()

In [ ]:
new_work = new_work.write()
work_qid = new_work.id
work_qid


### Download the existing entity and add another claim

In [ ]:
work_entity = wbi.item.get(entity_id=work_qid)
work_entity.claims.add([
            datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                        ])
work_entity.write()

### We should also register the work with Hennings!

In [ ]:
def add_work_to_author(author_qid : str, work_qid : str):
    raise NotImplementedError('Need to investigate if this is wanted')
    work = wbi.item.get(entity_id=author_qid)
    claims_to_add = [
        datatypes.Item(value=work_qid, prop_nr=PROPS['author_of']) 
                     ]
    work.claims.add(claims_to_add)
    work.write()

## Make a new edition entry, referencing the work

In [ ]:
def make_edition(author_qid : str, title : str, author : Author, year :int, 
                work_qid : Optional[str], url : Optional[str] = None): 
    language = ENTITIES['German']

    new_edition = wbi.item.new()
    new_edition.labels.set('de', f'{title} (Erstausgabe von {str(year)})')
    # Set a default label too
    new_edition.labels.set('mul', f'{title} (first edition, {str(year)})')

    new_edition.descriptions.set('en', 
                        f'{str(year)} edition of the literary work of fiction by {author.get_name('en')}')
    new_edition.descriptions.set('de', 
                        f'Ausgabe von {str(year)} des fiktionalen literarischen Werks von {author.get_name('de')}')

    new_edition.claims.add([
        datatypes.Item(value=ENTITIES['edition'], prop_nr=PROPS['instance_of']), 
        datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
        datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
        datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
        datatypes.Item(value=language, prop_nr=PROPS['language'])
                            ])   

    if work_qid: 
        new_edition.claims.add([ datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of'])
                               ])
        
    if url:
        new_work.claims.add([
            datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                        ])

    return new_edition


In [ ]:
print(author_qid)
print(title)
print(work_qid)
print(author.get_name())
print(url)

In [ ]:
new_edition = make_edition(author_qid = author_qid, year = year,
                           author = author,
                        title = title, 
                        work_qid = work_qid, url=url)
new_edition.get_json()

In [ ]:
new_edition = new_edition.write()
edition_qid = new_edition.id
edition_qid

In [ ]:
def make_gutenberg_edition(author_qid : str, title : str, author : Author, source : str,
                           work_qid : Optional[str] = None, edition_qid : Optional[str] = None, 
                           url : Optional[str] = None, year : Optional[int]=None, pg_id : Optional[str] = None): 

    language = ENTITIES['German']

    new_edition = wbi.item.new()

    if source == 'PG-DE':
        gb_edition_descr = {'en': 'Projekt Gutenberg-DE edition', 
                          'de': 'Projekt Gutenberg-DE Edition'}
    elif source == 'PG-US':
        gb_edition_descr = {'en': 'Project Gutenberg edition', 
                          'de': 'Project Gutenberg Edition'}
    else:
        raise Exception(f"source must be one of 'PG-DE' or 'PG-US'")

    new_edition.labels.set('de', title + f' ({gb_edition_descr['de']})')
    # Set a default label too
    new_edition.labels.set('mul', title + f' ({gb_edition_descr['en']})')

    new_edition.descriptions.set('en', 
                f'{gb_edition_descr['en']} of the literary work of fiction by {author.get_name('de')}')
    new_edition.descriptions.set('de', 
                f'{gb_edition_descr['de']} des fiktionalen literarischen Werks von {author.get_name('de')}')

    new_edition.claims.add([
        datatypes.Item(value=ENTITIES['edition'], prop_nr=PROPS['instance_of']), 
        datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of']),
        datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
        datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
        # We're not sure about the year, so let's leave it out
        #datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
        datatypes.Item(value=language, prop_nr=PROPS['language'])
                            ])

    if work_qid: 
        new_edition.claims.add([ datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of'])
                               ])
        
    if edition_qid: 
        new_edition.claims.add([ datatypes.Item(value=edition_qid, prop_nr=PROPS['based_on'])
                               ])   
    if url:
        new_edition.claims.add([
            datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                        ])
        
    if year:
        new_edition.claims.add([
            datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
                        ])
                  # already made sure its a string in the function header, but let's be robust
    if pg_id and (source == 'PG-US'):
            new_edition.claims.add([
                datatypes.ExternalID(value=pg_id, prop_nr=PROPS['project_gb_ebook_id']),
                        ])
        
    return new_edition



In [ ]:
source = work['source']
pg_id = work['gutenberg_id']
print(author_qid)
print(title)
print(work_qid)
print(edition_qid)
print(url)
print(pg_id)
print(type(pg_id))
print(source)
print(pg_id and (source == 'PG-US'))

In [ ]:
new_gb_edition = make_gutenberg_edition(author_qid = author_qid, 
                           title = title, 
                           author = author, 
                           work_qid= work_qid,
                           edition_qid = edition_qid, 
                           pg_id=pg_id, source=source)
new_gb_edition.get_json()

In [ ]:
new_gb_edition = new_gb_edition.write()
gb_edition_qid = new_gb_edition.id
gb_edition_qid

In [ ]:
def add_editions_to_work(work_qid : str, edition_qids : list[str]):
    work = wbi.item.get(entity_id=work_qid)
    claims_to_add = [datatypes.Item(value=edition_qid, prop_nr=PROPS['has_edition_or_translation']) 
                     for edition_qid in edition_qids]
    work.claims.add(claims_to_add)
    work.write()


In [ ]:
add_editions_to_work(work_qid=work_qid, edition_qids=[edition_qid, gb_edition_qid])

## Find these new works using SPARQL

In [ ]:
print(make_sparql_authors_works(author_qid))

### Author now has these editions on wikidata:

In [ ]:
result = wbi_helpers.execute_sparql_query(make_sparql_authors_works(author_qid), 
                                          user_agent=wbi_config['USER_AGENT'])
result